# 07 — Transferencia y ajuste de modelos preentrenados

Reutilizaremos un encoder con preentrenamiento autosupervisado documentado. Primero congelaremos sus pesos y entrenaremos una cabeza clasificadora sencilla; después estudiaremos un ajuste limitado si el presupuesto lo permite. El checkpoint concreto queda pendiente de selección y justificación.

Un modelo convencional supervisado de ImageNet puede servir como contraste opcional. Estar preentrenado no significa haber aprendido de forma autosupervisada. Se documentarán el objetivo original, los datos externos y el preprocesamiento esperado.

> **Plan pendiente de implementación.** Este cuaderno contiene explicaciones y propuestas. No contiene código ejecutado ni resultados de entrenamiento.

## Requisito confirmado: aplicar aprendizaje autosupervisado

El mínimo incluye un encoder con preentrenamiento autosupervisado documentado, además de explicar transferencia y fine-tuning. Seleccionar un checkpoint manejable con procedencia, objetivo, datos y preprocesamiento verificables; no reproducir su preentrenamiento completo. Ejecutar extracción de embeddings con encoder congelado + cabeza sencilla y una adaptación limitada si es viable. Si se usa un subconjunto por coste, fijarlo por grupos de train antes del ajuste y usarlo en ambos contrastes. Registrar memoria, tiempo y parámetros.

Un pequeño autoencoder de reconstrucción entrenado sin etiquetas puede ilustrar el objetivo autosupervisado, pero no basta para afirmar que se ha estudiado un modelo fundacional externo. ResNet supervisada queda como contraste opcional. Conectar los resultados con zero-shot en 11 sin atribuir a todo checkpoint capacidad zero-shot. El experimento aún está pendiente.

## Vía B · Lo habitual en aplicación: reutilizar representaciones

Interpretamos esta pista de clase como la posibilidad de resolver el problema reutilizando un modelo, frente a entrenarlo todo desde cero. Es una práctica habitual, no una obligación universal ni una garantía de mejores resultados. Primero: encoder congelado + cabeza entrenable. Después: fine-tuning controlado. Comparar con scratch manteniendo los grupos de producto y explicitando diferencias de preprocesamiento y presupuesto.

**Preentrenamiento y autosupervisión no son sinónimos.** Un checkpoint convencional de clasificación ImageNet puede estar preentrenado con etiquetas humanas; no se etiquetará como self-supervised. Una alternativa autosupervisada aprende representaciones con tareas construidas a partir de los propios datos; [SimCLR](https://arxiv.org/abs/2002.05709) proporciona un ejemplo contrastivo visual. En la alternativa autosupervisada obligatoria, identificar checkpoint, datos, objetivo y protocolo original, sin suponer que deba reproducirse su costoso preentrenamiento aquí.

Los dos tipos de encoder pueden producir embeddings visuales para una cabeza clasificadora. Esos vectores no son word vectors ni generación de imágenes. Esta vía pretende explicar transferencia, congelación y adaptación; el rendimiento es evidencia que interpretar, no el único objetivo.

## Protocolo común del proyecto

Primero definimos las prendas que queremos clasificar. Después aplicamos los mismos controles de calidad a todas las fotos y conservamos **todas las que los superan**, aunque unas clases tengan más ejemplos. No se fija un número total ni se igualan las clases con cupos.

Las particiones se construyen sobre todos los grupos de producto elegibles. Los productos ya utilizados durante el desarrollo conservan su exposición: un producto de train no se convierte en un supuesto test nuevo. Los grupos nuevos completan aproximadamente 70/15/15 por clase. Se publican los recuentos reales.

**Antes de recortar train por coste, hay que medirlo.** El baseline usa todos los productos elegibles; 04 registra una época piloto y la estimación de tiempo/memoria. Si fuese necesario reducir datos en otro experimento, se justificaría con esa medición, solo en train y conservando validation/test. Una fotografía fuera de una muestra de cálculo no sería una fotografía de mala calidad.

La referencia es `config/experimental_protocol_v1.json`: macro-F1 primaria, accuracy y resultados por clase; pérdida inicial sin pesos, sin aumentos; normalización solo con train a la resolución de entrada. Tres semillas y coste computacional. Los pesos de clase se estudian como ablación: con frecuencias desiguales ya no suponemos que sean casi uno. El test predictivo se reserva hasta congelar las decisiones. Las métricas describen las 26 clases elegidas tras los controles, no todo el catálogo.

### H3 · Aislar el efecto de los pesos preentrenados

Comparar la misma arquitectura de backbone, resolución y transformación de entrada con inicialización aleatoria y preentrenada. Publicar por separado encoder congelado + cabeza y fine-tuning: sus parámetros entrenables y presupuestos son distintos. Mantener train/validation, semilla, límite de épocas y criterio de selección. Informar coste local y reconocer el coste/datos externos del preentrenamiento. Una CNN pequeña a 64×64 frente a ResNet preentrenada a 224×224 es una comparación aplicada útil, pero no una estimación causal aislada del efecto del preentrenamiento.

**Filtro y reutilización.** En 02 se aplica el mismo criterio visual fijo a train, validation y test; las fotos conservadas no cambian de partición. El test se reserva para evaluación predictiva, aunque haya pasado controles e inspección visual. Se guardarán resultados y predicciones para recalcular métricas sin repetir ajustes. Cualquier extracción o entrenamiento costoso nuevo requerirá una activación explícita; este cuaderno todavía no implementa ese flujo.

## Objetivos

- Identificar un checkpoint autosupervisado manejable y documentar su procedencia.
- Guardar las representaciones del encoder congelado para reutilizarlas sin extraerlas otra vez.
- Comparar encoder congelado + cabeza y ajuste limitado, indicando los parámetros entrenables.
- Respetar el preprocesamiento del checkpoint y explicar el modo de BatchNorm si lo utiliza.
- Distinguir una comparación aplicada entre modelos del contraste controlado que aísla la inicialización preentrenada.

## Pasos previstos

1. Seleccionar y documentar encoder, objetivo y transformaciones.
2. Extraer y guardar embeddings una sola vez por combinación de datos, encoder y preprocesamiento.
3. Entrenar la cabeza sobre las representaciones guardadas.
4. Medir el coste y estudiar un ajuste limitado si resulta viable.
5. Comparar resultados con entrenamiento desde cero y explicar las diferencias de datos externos y presupuesto.

## Entradas, salidas y material de clase

| | |
|---|---|
| **Entradas** | Particiones de 02 y preprocesamiento documentado del checkpoint |
| **Salidas** | Embeddings guardados, métricas, predicciones y configuraciones en results/ |
| **Material de clase** | `MLLB_TransferLearning_Image_and-Text.pptx`, `MLLB_Keras_transfer_learning_v2.ipynb`, `pytorch_transfer_learning_colab.ipynb` |